# Выполнение ЛР №6: "Поиск ассоциативных правил" 

## Подключение библиотек

In [ ]:
import os
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori as mlxtend_apriori, association_rules as mlxtend_association_rules

pd.set_option('display.max_rows', None)

## Задание

Реализовать  на  любом языке программирования алгоритм поиска ассоциативных правил: Apriori

Для  проверки  корректности 
алгоритма на любом предлагаемом датасете предлагается 
применить библиотеку mlxtend  языка Python.

## Изучение датасета

In [ ]:
# Загрузка датасета Anketa1

path_anketa = os.path.join(os.getcwd(), 'Задание', 'dataset', 'Anketa1.txt')
df_raw = pd.read_csv(path_anketa, sep='\t', decimal=',', encoding='cp1251', encoding_errors='replace')
df_raw.head(10)

In [ ]:
# изучение данных
print('Размер:', df_raw.shape)
print('\nТипы и пропуски:')
print(df_raw.dtypes)

In [ ]:
# Уникальные значения категориальных признаков (для бинаризации)
cat_cols = [
    'Социальный статус',
    'Социальный статус супруга(и)',
    'Образование',
    'Наличие личного автомобиля',
    'Жилая недвижимость в собственности',
    'Наличие кредитов'
]
cat_uniques = {col: df_raw[col].astype(str).unique() for col in cat_cols}
df_cat_uniques = pd.DataFrame(dict([(col, pd.Series(vals)) for col, vals in cat_uniques.items()]))
display(df_cat_uniques)

## Подготовка DataFrame для ассоциативных правил

In [ ]:
# Удаляем идентификаторы
cols_drop = ['КодАнкеты', 'Фамилия', 'Имя', 'Отчество']
df = df_raw.drop(columns=[c for c in cols_drop if c in df_raw.columns], errors='ignore')

# Унификация да/нет (приведём к одному регистру)
binary_cols = ['Наличие личного автомобиля', 'Наличие кредитов']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df.loc[df[col].str.contains('да', na=False), col] = 'да'
        df.loc[df[col].str.contains('нет', na=False), col] = 'нет'

df.head()

In [ ]:
# Дискретизация числовых признаков (квартили или фиксированные границы)
def discretize(s, n_bins=3, labels=None):
    if labels is None:
        labels = [f'Q{i+1}' for i in range(n_bins)]
    # без labels — узнаём реальное число интервалов
    binned = pd.qcut(s, q=n_bins, duplicates='drop')
    n_actual = binned.cat.categories.size
    use_labels = labels[:n_actual] if len(labels) >= n_actual else [f'Q{i+1}' for i in range(n_actual)]
    return pd.qcut(s, q=n_bins, labels=use_labels, duplicates='drop')


num_cols = {
    'Сумма кредита, руб#': ['Малая', 'Средняя', 'Высокая'],
    'Количество лет проживания в регионе': ['Мало', 'Среднее', 'Много'],
    'Стаж работы, лет': ['Мало', 'Среднее', 'Много'],
    'Личный доход в месяц после налогооблажения': ['Низкий', 'Средний', 'Высокий'],
    'Рыночная стоимость автомобиля, руб#': ['Низкая', 'Средняя', 'Высокая'],  # 0 попадает в низ
    'Рыночная стоимость недвижимости, руб#': ['Низкая', 'Средняя', 'Высокая'],
}

for col, lab in num_cols.items():
    if col in df.columns:
        df[col + '_кат'] = discretize(df[col].astype(float), n_bins=len(lab), labels=lab)
            
df.head(10)

In [ ]:
# Собираем все категориальные колонки для транзакций // Не стал брать колонку "Социальный статус" т.к там одно значение
cat_cols = [
    'Социальный статус супруга(и)', 'Образование',
    'Наличие личного автомобиля', 'Жилая недвижимость в собственности', 'Наличие кредитов',
    'Возврат кредита'
]
# Добавляем дискретизированные
cat_cols += [c for c in df.columns if c.endswith('_кат')]

df_cat = df[cat_cols]
df_cat.head()

## Применение алгоритма Apriori

### Собственная реализация

In [ ]:
from apriori import * 

min_support_val = 0.5
min_confidence_val = 0.6

In [ ]:
df = df_cat

print('DataFrame:')
display( df.head())

transactions = dataframe_to_transactions(df, binary=False, prefix_with_column=True)
n_tarn = len(transactions)


print('\nТранзакции:')
for i, t in enumerate(transactions, start=1):
    print(f'T{i}:', t)

frequent_counts = apriori(transactions, min_support=min_support_val)
supports = compute_supports(frequent_counts, n_tarn)

print(f'\nЧастые наборы (support) : {len(supports)} шт.')
for itemset, sup in supports.items():
    print(itemset, '=>', round(sup, 3))

rules = generate_association_rules(frequent_counts, n_tarn, min_confidence=min_confidence_val)
print(f'\nАссоциативные правила (support, confidence): {len(rules)} шт.')
for r in rules:
    print(f'{r.antecedent} -> {r.consequent} (support={r.support:.3f}, conf={r.confidence:.3f})')

### Проверка с помощью mlxtend

In [ ]:
# Преобразование в формат "транзакций" для mlxtend: одна колонка = один предмет (признак=значение)
# Каждая строка — одна анкета (транзакция), колонки — бинарные признаки вида "Образование=высшее"
transactions = []
for _, row in df_cat.iterrows():
    trans = [f"{col} = {val}" for col, val in row.items()]
    transactions.append(trans)

te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_transactions = pd.DataFrame(te_ary, columns=te.columns_).astype(bool) 

print('Размер матрицы транзакций:', df_transactions.shape)
df_transactions.head(10)

In [ ]:
frequent_itemsets_mlxtend = mlxtend_apriori(df_transactions, min_support=min_support_val, use_colnames=True)

print(f'Частые наборы (mlxtend, min_support={min_support_val}): {len(frequent_itemsets_mlxtend)} шт.')
for index, row in frequent_itemsets_mlxtend.sort_values(by='support').iterrows():
    print(row['itemsets'], '=>', round(row['support'], 3))

rules_mlxtend = mlxtend_association_rules(frequent_itemsets_mlxtend, metric="confidence", min_threshold=min_confidence_val)

print(f'\nАссоциативные правила (mlxtend, min_confidence={min_confidence_val}): {len(rules_mlxtend)} шт.')
for index, r in rules_mlxtend.iterrows():
    print(f'{r.antecedents} -> {r.consequents} (support={r.support:.3f}, conf={r.confidence:.3f})')

In [ ]:
# Сравнение результатов собственной реализации и mlxtend

rules_custom_set = {(r.antecedent, r.consequent): (r.support, r.confidence) for r in rules}

rules_mlxtend_set = {}
for _, row in rules_mlxtend.iterrows():
    ant = row['antecedents']
    cons = row['consequents']
    sup = row['support']
    conf = row['confidence']
    rules_mlxtend_set[(ant, cons)] = (sup, conf)

common = set(rules_custom_set) & set(rules_mlxtend_set)
only_custom = set(rules_custom_set) - set(rules_mlxtend_set)
only_mlxtend = set(rules_mlxtend_set) - set(rules_custom_set)

print('Сравнение ассоциативных правил:')
print(f'  Собственная реализация: {len(rules)} правил')
print(f'  mlxtend:                {len(rules_mlxtend)} правил')
print(f'  Совпадают:              {len(common)} правил')
if only_custom:
    print(f'  Только в собственной:   {len(only_custom)}')
if only_mlxtend:
    print(f'  Только в mlxtend:       {len(only_mlxtend)}')
if len(common) == len(rules) == len(rules_mlxtend) and not only_custom and not only_mlxtend:
    print('\nРезультаты полностью совпадают. Собственная реализация Apriori корректна.')

## Анализ результатов поиска ассоциативных правил

Прежде чем интерпретировать правила, зафиксируем опорные точки. Целевая переменная **«Возврат кредита = 1»** имеет support = 0.759 — это означает, что ~76% наблюдений в выборке относятся к классу «кредит возвращён».

### Анализ частых наборов

Наблюдается явный кластер взаимосвязанных признаков вокруг «бедности активов»:
* «Наличие кредитов = нет» (0.88) — доминирующий признак, встречается в 88% записей
* «Возврат кредита = 1» (0.759)
* «Рыночная стоимость недвижимости = Низкая» (0.663) и «Наличие кредитов = нет + Возврат кредита = 1» (0.663)

Все 3-элементные наборы замкнуты вокруг одной тройки (с support = 0.53): 
* Низкая стоимость недвижимости, 
* Нет кредитов, 
* Нет жилья в собственности

Это говорит о том, что данные признаки образуют один плотный кластер «клиентов без активов», а не несколько независимых паттернов.

### Анализ ассоциативных правил

#### Правила с confidence = 1.0 (детерминированные)
```
{Жилья в собственности = Нет} → {Стоимость недвижимости = Низкая}  (sup=0.602, conf=1.000)
{Нет автомобиля} → {Стоимость автомобиля = Низкая}                 (sup=0.542, conf=1.000)
```

* Эти правила являются прямым следствием логики кодирования данных. Если у клиента нет имущества, его рыночная стоимость категоризируется как «Низкая» (фактически — нулевая). 
* Это артефакт feature engineering, а не содержательный паттерн. 
* В продуктивной модели такие признаки избыточны и должны быть консолидированы.

#### Ключевые правила для бизнес-задачи

Наиболее интересны правила с участием целевой переменной:

| Правило | Support | Confidence | Lift\* |
| --- | --- | --- | --- |
| {Нет кредитов} → {Возврат = 1} | 0.663 | 0.753 | ~0.99 |
| {Стоимость недвижимости = Низкая} → {Возврат = 1} | 0.542 | 0.818 | ~1.08 |
| {Стоимость автомобиля = Низкая} → {Возврат = 1} | 0.518 | 0.768 | ~1.01 |
| {Возврат = 1} → {Нет кредитов} | 0.663 | 0.873 | ~0.99 |

`Lift = conf / P(consequent). P(Возврат=1) ≈ 0.759, P(Нет кредитов) ≈ 0.880`

lift для большинства правил с «Возврат кредита = 1» близок к 1.0, что означает — предпосылки практически не улучшают предсказание относительно базовой частоты класса.

### Вывод

Полученные правила подтверждают интуитивно тривиальный профиль: клиенты без значимых активов и без кредитной истории в большинстве своём возвращают небольшие кредиты. Для построения скоринговой модели данный набор правил в текущем виде имеет ограниченную практическую ценность и требует доработки признакового пространства.